# 04. SE(3)-Invariant GNN Pose Encoder + Equivariant Decoder

**연구의 핵심 모듈**.

## 설계
- **Encoder** (Teacher): pose graph (N nodes, 3D coords) → invariant latent z (32-dim).
  - EGNN (Satorras et al. 2021) 또는 e3nn 기반.
  - **Invariant** scalar feature만 readout.
- **Decoder** (per-robot Student): z + robot graph → joint angles.
  - z를 노드별 feature로 broadcast.
  - GNN message passing (kinematic edges).
  - 각 노드에서 1-DOF angle output.

## 검증
- Equivariance test (encoder가 회전/평행이동에 invariant인지).
- Reconstruction loss 감소 확인.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
PROJECT_ROOT = '/content/drive/MyDrive/openvla-pose'

!pip install -q torch torch_geometric e3nn

## 1. EGNN 레이어 — 가장 단순한 SE(3) equivariant GNN

Satorras et al. "E(n) Equivariant Graph Neural Networks" (ICML 2021).
노드 좌표 `x`와 invariant feature `h`를 동시에 업데이트:
```
m_ij = phi_e(h_i, h_j, ||x_i - x_j||^2)
x_i' = x_i + sum_j (x_i - x_j) * phi_x(m_ij)   ← equivariant
h_i' = phi_h(h_i, sum_j m_ij)                  ← invariant
```

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class EGNNLayer(nn.Module):
    def __init__(self, h_dim, edge_dim=0, hidden=128):
        super().__init__()
        self.phi_e = nn.Sequential(
            nn.Linear(2*h_dim + 1 + edge_dim, hidden), nn.SiLU(),
            nn.Linear(hidden, hidden), nn.SiLU(),
        )
        self.phi_x = nn.Sequential(
            nn.Linear(hidden, hidden), nn.SiLU(),
            nn.Linear(hidden, 1),
        )
        self.phi_h = nn.Sequential(
            nn.Linear(h_dim + hidden, hidden), nn.SiLU(),
            nn.Linear(hidden, h_dim),
        )

    def forward(self, h, x, edge_index, edge_attr=None):
        # h: [N, h_dim], x: [N, 3], edge_index: [2, E]
        i, j = edge_index
        rel = x[i] - x[j]                      # [E, 3]
        d2 = (rel ** 2).sum(-1, keepdim=True)  # [E, 1]
        m_in = [h[i], h[j], d2]
        if edge_attr is not None:
            m_in.append(edge_attr)
        m = self.phi_e(torch.cat(m_in, dim=-1))     # [E, hidden]
        # Equivariant coordinate update
        coord_w = self.phi_x(m)                      # [E, 1]
        x_msg = rel * coord_w                        # [E, 3]
        N = h.size(0)
        x_agg = torch.zeros_like(x).index_add_(0, i, x_msg)
        x_new = x + x_agg / max(1, edge_index.size(1) // N)
        # Invariant scalar update
        h_agg = torch.zeros(N, m.size(-1), device=h.device).index_add_(0, i, m)
        h_new = h + self.phi_h(torch.cat([h, h_agg], dim=-1))
        return h_new, x_new

print('EGNNLayer defined')

## 2. Pose Encoder — invariant readout 32-dim z

In [ ]:
class PoseEncoder(nn.Module):
    def __init__(self, n_layers=4, h_dim=64, z_dim=32, n_node_types=21):
        super().__init__()
        self.embed = nn.Embedding(n_node_types, h_dim)
        self.layers = nn.ModuleList([EGNNLayer(h_dim) for _ in range(n_layers)])
        # Invariant readout: average of h, then MLP
        self.readout = nn.Sequential(
            nn.Linear(h_dim, 128), nn.SiLU(),
            nn.Linear(128, z_dim),
        )

    def forward(self, x, node_type, edge_index):
        h = self.embed(node_type)
        for L in self.layers:
            h, x = L(h, x, edge_index)
        z = self.readout(h.mean(0))   # [z_dim]
        return z

print('PoseEncoder defined')

## 3. Robot Decoder — z → joint angles

로봇별로 별도 인스턴스. 노드 수가 robot DOF에 맞게 다름.
구조는 기구학적 chain (직렬 연결 그래프).

In [ ]:
class RobotDecoder(nn.Module):
    def __init__(self, z_dim=32, n_joints=7, h_dim=64, n_layers=3):
        super().__init__()
        self.n_joints = n_joints
        self.z_to_node = nn.Linear(z_dim, h_dim * n_joints)
        self.layers = nn.ModuleList([EGNNLayer(h_dim) for _ in range(n_layers)])
        self.head = nn.Sequential(
            nn.Linear(h_dim, 64), nn.SiLU(),
            nn.Linear(64, 1),
        )
        # serial chain: 0-1-2-...-(N-1)
        edges = []
        for i in range(n_joints - 1):
            edges += [(i, i+1), (i+1, i)]
        self.register_buffer('edge_index', torch.tensor(edges).t().contiguous())
        # initial joint positions along z-axis (placeholder)
        x_init = torch.zeros(n_joints, 3)
        x_init[:, 2] = torch.linspace(0, 1, n_joints)
        self.register_buffer('x_init', x_init)

    def forward(self, z):
        h = self.z_to_node(z).view(self.n_joints, -1)
        x = self.x_init.clone()
        for L in self.layers:
            h, x = L(h, x, self.edge_index)
        angles = self.head(h).squeeze(-1)
        return angles  # [n_joints]

print('RobotDecoder defined')

## 4. **Equivariance/Invariance 검증** — 학습 전에 반드시 통과해야 함

In [ ]:
def random_so3():
    """Random rotation matrix via QR."""
    A = torch.randn(3, 3)
    Q, R = torch.linalg.qr(A)
    return Q * torch.sign(torch.diag(R)).unsqueeze(0)

def check_invariance(encoder, n_trials=10, atol=1e-4):
    encoder.eval()
    failures = []
    with torch.no_grad():
        for _ in range(n_trials):
            N = 21
            x = torch.randn(N, 3)
            node_type = torch.arange(N)
            edge_index = torch.tensor([[i, (i+1)%N] for i in range(N)] + [[(i+1)%N, i] for i in range(N)]).t()
            z1 = encoder(x, node_type, edge_index)
            R = random_so3()
            t = torch.randn(3)
            x2 = x @ R.t() + t
            z2 = encoder(x2, node_type, edge_index)
            diff = (z1 - z2).abs().max().item()
            failures.append(diff)
    failures = torch.tensor(failures)
    print(f'Max |z - z_rot| over {n_trials} trials: mean={failures.mean():.2e}  max={failures.max():.2e}')
    return (failures < atol).all().item()

encoder = PoseEncoder()
ok = check_invariance(encoder)
print('PASS' if ok else 'FAIL — encoder가 SE(3) invariant가 아님')

## 5. 합성 데이터로 distillation training

실제 로봇 URDF + FK로 (joint_angles, joint_3d_positions) 생성 → encoder로 z → decoder로 다시 angles 복원.

In [ ]:
def fake_fk_chain(angles, link_len=0.1):
    """단순 2D 평면 chain — 실제 코드에선 pytorch-kinematics로 교체."""
    N = angles.shape[0]
    pos = torch.zeros(N + 1, 3)
    cum = 0.
    for i, a in enumerate(angles):
        cum = cum + a
        pos[i+1] = pos[i] + torch.tensor([torch.cos(cum)*link_len, torch.sin(cum)*link_len, 0.])
    return pos[1:]

torch.manual_seed(0)
encoder = PoseEncoder(n_node_types=7)
decoder = RobotDecoder(n_joints=7)
opt = torch.optim.Adam(list(encoder.parameters()) + list(decoder.parameters()), lr=1e-3)

edge_index = torch.tensor([[i, i+1] for i in range(6)] + [[i+1, i] for i in range(6)]).t()
node_type = torch.arange(7)

losses = []
for step in range(300):
    angles = (torch.rand(7) * 2 - 1) * 1.5
    pos = fake_fk_chain(angles)
    z = encoder(pos, node_type, edge_index)
    pred = decoder(z)
    loss = F.mse_loss(pred, angles)
    opt.zero_grad(); loss.backward(); opt.step()
    losses.append(loss.item())
    if step % 50 == 0:
        print(f'step {step:3d}  loss {loss.item():.4f}')

import matplotlib.pyplot as plt
plt.semilogy(losses); plt.xlabel('step'); plt.ylabel('MSE (log)'); plt.title('GNN distillation smoke test')
plt.show()

# 학습 후 invariance 재검증
print('\nPost-training invariance check:')
check_invariance(encoder)

## 6. 체크포인트 저장

In [ ]:
CKPT = f'{PROJECT_ROOT}/checkpoints/gnn_v0.pt'
os.makedirs(os.path.dirname(CKPT), exist_ok=True)
torch.save({
    'encoder': encoder.state_dict(),
    'decoder': decoder.state_dict(),
    'config': {'z_dim': 32, 'n_joints': 7},
}, CKPT)
print('Saved', CKPT)

## 7. Gradio 데모 — z 슬라이더로 잠재 포즈 공간 탐색

각 z 차원을 슬라이더로 조절 → decoder가 즉시 robot pose 그림.
"이 차원이 무슨 의미인지" 직관적으로 보는 도구.


In [ ]:
!pip install -q gradio

import gradio as gr
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D  # noqa

Z_DIM = 32
N_SLIDERS = 8  # 처음 8차원만 슬라이더로

def explore(*z_vals):
    z = torch.zeros(Z_DIM)
    for i, v in enumerate(z_vals):
        z[i] = v
    decoder.eval()
    with torch.no_grad():
        angles = decoder(z)
    pos = fake_fk_chain(angles)  # [N, 3]
    fig = plt.figure(figsize=(5, 5))
    ax = fig.add_subplot(111, projection='3d')
    p = pos.numpy()
    ax.scatter(p[:, 0], p[:, 1], p[:, 2], s=50, c='C0')
    for i in range(len(p) - 1):
        ax.plot(p[i:i+2, 0], p[i:i+2, 1], p[i:i+2, 2], 'k-', lw=2)
    ax.set_xlim(-1, 1); ax.set_ylim(-1, 1); ax.set_zlim(-0.5, 0.5)
    ax.set_title(f'Robot pose from z[:{N_SLIDERS}]')
    angle_str = '  '.join(f'{a:+.2f}' for a in angles.numpy())
    return fig, f'angles: {angle_str}'

inputs = [gr.Slider(-2.0, 2.0, value=0.0, label=f'z[{i}]') for i in range(N_SLIDERS)]
outputs = [gr.Plot(label='Decoded skeleton'), gr.Textbox(label='Joint angles')]

demo = gr.Interface(
    fn=explore,
    inputs=inputs, outputs=outputs,
    title='GNN Pose Space Explorer',
    description='Move sliders → see decoded robot pose. Other z dims are 0.',
    live=True,
)
demo.launch(share=True, debug=False)
